1 — Import Libraries & Define Project Paths

In [ ]:
# ============================================================
# CELL 1 - IMPORT LIBRARIES AND DEFINE PROJECT PATHS
# Purpose:
# Initialize required libraries and define input/output paths.
# ============================================================

from pathlib import Path
from collections import Counter
from itertools import zip_longest

import csv
import hashlib
import os

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Locate the project root directory
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "data").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

assert (PROJECT_ROOT / "data").is_dir(), (
    "Project root not found. Open the UMDAC_datathon folder."
)


# ------------------------------------------------------------
# 2. Define the original dataset location
# ------------------------------------------------------------

RAW_DIR = PROJECT_ROOT / "data" / "raw"

RAW_CSV_PATH = (
    RAW_DIR / "swift_payments_raw_export.csv"
)

RAW_EXCEL_PATH = (
    RAW_DIR
    / "swift_umdac_datathon_2026_fictitious_payment_data.xlsx"
)


# ------------------------------------------------------------
# 3. Define output directories
# ------------------------------------------------------------

CLEAN_DIR = PROJECT_ROOT / "data" / "cleaned"
REPORT_DIR = PROJECT_ROOT / "outputs" / "tables"

CLEAN_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 4. Define output file paths
# ------------------------------------------------------------

CLEAN_CSV_PATH = (
    CLEAN_DIR / "swift_payments_cleaned.csv"
)

TEMP_CSV_PATH = (
    CLEAN_DIR / "swift_payments_cleaned.tmp.csv"
)

CLEANING_LOG_PATH = (
    REPORT_DIR / "cleaning_actions_log.csv"
)

VALIDATION_REPORT_PATH = (
    REPORT_DIR / "cleaning_validation_report.csv"
)


# ------------------------------------------------------------
# 5. Configure batch processing
# ------------------------------------------------------------

CHUNK_SIZE = 25_000
CSV_ENCODING = "utf-8-sig"


# ------------------------------------------------------------
# 6. Verify source files
# ------------------------------------------------------------

assert RAW_CSV_PATH.is_file(), (
    f"Raw CSV file not found: {RAW_CSV_PATH}"
)

assert RAW_EXCEL_PATH.is_file(), (
    f"Original Excel file not found: {RAW_EXCEL_PATH}"
)


# ------------------------------------------------------------
# 7. Display setup information
# ------------------------------------------------------------

print("=" * 65)
print("SWIFT DATATHON - DATA CLEANING SETUP")
print("=" * 65)

print("Project root:", PROJECT_ROOT)
print("Raw CSV:", RAW_CSV_PATH)
print("Original Excel:", RAW_EXCEL_PATH)
print("Cleaned CSV:", CLEAN_CSV_PATH)
print("Chunk size:", CHUNK_SIZE)

print("\nSetup completed successfully.")

2 — Define Original Schema & Cleaning Policy

In [ ]:
# ============================================================
# CELL 2 - DEFINE ORIGINAL SCHEMA AND CLEANING POLICY
# Purpose:
# Protect the original dataset and define safe transformations.
# ============================================================

ORIGINAL_COLUMNS = [
    "DISCLAIMER",
    "EPOCH_SECONDS",
    "CURRENCY_CODE",
    "USD_AMOUNT",
    "ORDERING_BIC8",
    "BENEFICIARY_BIC8",
    "BIC8_PATH",
    "TOTAL_DURATION",
    "CONTINUITY_QFLAG",
    "CURRENCY_CONVERSION_COUNT",
    "PARTICIPANT_COUNT",
    "TRANSACTION_TYPE_CODE",
    "ORDERING_ACCOUNT",
    "BENEFICIARY_ACCOUNT",
    "ORDERING_DETAILS",
    "ORDERING_TOKENS",
    "BENEFICIARY_DETAILS",
    "BENEFICIARY_TOKENS",
    "ORDERING_TYPE",
    "BENEFICIARY_TYPE",
    "ANOMALY_TYPE"
]


# ------------------------------------------------------------
# 1. Define categorical columns for standardization
# ------------------------------------------------------------

STANDARDIZE_COLUMNS = [
    "CURRENCY_CODE",
    "CONTINUITY_QFLAG",
    "TRANSACTION_TYPE_CODE",
    "ORDERING_TYPE",
    "BENEFICIARY_TYPE",
    "ANOMALY_TYPE"
]


# ------------------------------------------------------------
# 2. Define numeric analysis columns
# ------------------------------------------------------------

NUMERIC_MAPPING = {
    "EPOCH_SECONDS": "EPOCH_SECONDS_NUM",
    "USD_AMOUNT": "USD_AMOUNT_NUM",
    "TOTAL_DURATION": "TOTAL_DURATION_NUM",
    "PARTICIPANT_COUNT": "PARTICIPANT_COUNT_NUM",
    "CURRENCY_CONVERSION_COUNT": "FX_COUNT_NUM"
}


# ------------------------------------------------------------
# 3. Define expected categories
# ------------------------------------------------------------

EXPECTED_CATEGORIES = {
    "CONTINUITY_QFLAG": {"Y", "N"},

    "TRANSACTION_TYPE_CODE": {
        "DIRECT",
        "INDIRECT",
        "COVERED"
    },

    "ORDERING_TYPE": {"B", "C"},

    "BENEFICIARY_TYPE": {"B", "C"},

    "ANOMALY_TYPE": {
        "ROMANCE_SCAM",
        "CEO_SCAM",
        "ACCOUNT_REPLACEMENT_SCAM",
        "FAT_FINGER_MISTAKE",
        "DUPLICATE_PAYMENT"
    }
}


# ------------------------------------------------------------
# 4. Define the data preservation policy
# ------------------------------------------------------------

CLEANING_POLICY = {
    "DELETE_ROWS": False,
    "DROP_DUPLICATES": False,
    "REMOVE_OUTLIERS": False,
    "FILL_MISSING_VALUES": False,
    "MODIFY_ORIGINAL_COLUMNS": False,
    "REMOVE_ROUTING_GAPS": False,
    "RENAME_ANOMALY_LABELS": False,
    "ADD_DERIVED_COLUMNS": True
}


# ------------------------------------------------------------
# 5. Validate the source schema
# ------------------------------------------------------------

actual_columns = pd.read_csv(
    RAW_CSV_PATH,
    encoding=CSV_ENCODING,
    nrows=0
).columns.tolist()

assert actual_columns == ORIGINAL_COLUMNS, (
    "Original dataset columns do not match the expected schema."
)


# ------------------------------------------------------------
# 6. Display policy
# ------------------------------------------------------------

print("=" * 65)
print("DATA CLEANING POLICY")
print("=" * 65)

for policy, enabled in CLEANING_POLICY.items():
    print(f"{policy}: {enabled}")

print("\nOriginal columns:", len(ORIGINAL_COLUMNS))
print("Categorical fields:", len(STANDARDIZE_COLUMNS))
print("Numeric fields:", len(NUMERIC_MAPPING))

print("\nSchema validation passed.")

3 — Define Standardization & Feature Engineering Function

In [ ]:
# ============================================================
# CELL 3 - DEFINE SAFE DATA CLEANING FUNCTION
# Purpose:
# Preserve source data while creating standardized categories,
# numeric analysis fields, and routing-related derived features.
# ============================================================

def count_observed_bics(path):

    # Count visible BIC nodes without changing the path.

    if not isinstance(path, str):
        return 0

    if not path.strip():
        return 0

    nodes = path.split(";")

    return sum(
        1
        for node in nodes
        if node.strip() not in ("", "...")
    )


def clean_payment_chunk(df, starting_row):

    # Create an independent copy of the original chunk.
    cleaned = df.copy()


    # --------------------------------------------------------
    # 1. Preserve the original Excel row number
    # --------------------------------------------------------

    cleaned["SOURCE_ROW_NUMBER"] = np.arange(
        starting_row,
        starting_row + len(cleaned),
        dtype=np.int64
    )


    # --------------------------------------------------------
    # 2. Create standardized categorical columns
    # --------------------------------------------------------

    for col in STANDARDIZE_COLUMNS:

        cleaned[col + "_STD"] = (
            cleaned[col]
            .astype(str)
            .str.strip()
            .str.upper()
        )


    # --------------------------------------------------------
    # 3. Create numeric analysis columns
    # --------------------------------------------------------

    for source_col, target_col in NUMERIC_MAPPING.items():

        raw_values = cleaned[source_col].str.strip()

        cleaned[target_col] = pd.to_numeric(
            raw_values,
            errors="coerce"
        )


    # --------------------------------------------------------
    # 4. Convert Unix epoch seconds to UTC datetime
    # --------------------------------------------------------

    cleaned["DATETIME_UTC"] = pd.to_datetime(
        cleaned["EPOCH_SECONDS_NUM"],
        unit="s",
        utc=True,
        errors="coerce"
    )

    cleaned["DATE_UTC"] = (
        cleaned["DATETIME_UTC"]
        .dt.strftime("%Y-%m-%d")
        .fillna("")
    )

    cleaned["HOUR_UTC"] = (
        cleaned["DATETIME_UTC"]
        .dt.hour
        .astype("Int64")
    )

    cleaned["DAY_OF_WEEK_UTC"] = (
        cleaned["DATETIME_UTC"]
        .dt.day_name()
        .fillna("")
    )

    cleaned["DATETIME_UTC"] = (
        cleaned["DATETIME_UTC"]
        .dt.strftime("%Y-%m-%dT%H:%M:%SZ")
        .fillna("")
    )


    # --------------------------------------------------------
    # 5. Create party relationship categories
    # --------------------------------------------------------

    ordering_type = cleaned["ORDERING_TYPE_STD"]
    beneficiary_type = cleaned["BENEFICIARY_TYPE_STD"]

    valid_party_types = (
        ordering_type.isin(["B", "C"])
        &
        beneficiary_type.isin(["B", "C"])
    )

    cleaned["PARTY_PAIR"] = "UNKNOWN"

    cleaned.loc[
        valid_party_types,
        "PARTY_PAIR"
    ] = (
        ordering_type.loc[valid_party_types]
        + "_"
        + beneficiary_type.loc[valid_party_types]
    )


    # --------------------------------------------------------
    # 6. Identify routing path gaps
    # --------------------------------------------------------

    path_values = cleaned["BIC8_PATH"].astype(str)

    missing_path = path_values.str.strip().eq("")

    has_path_gap = path_values.str.contains(
        "...",
        regex=False,
        na=False
    )

    cleaned["HAS_PATH_GAP"] = "N"

    cleaned.loc[has_path_gap, "HAS_PATH_GAP"] = "Y"

    cleaned.loc[missing_path, "HAS_PATH_GAP"] = "UNKNOWN"


    # --------------------------------------------------------
    # 7. Count observable BIC nodes
    # --------------------------------------------------------

    cleaned["OBSERVED_BIC_COUNT"] = (
        path_values.map(count_observed_bics)
    )


    # --------------------------------------------------------
    # 8. Create a non-destructive anomaly indicator
    # --------------------------------------------------------

    anomaly_values = cleaned["ANOMALY_TYPE_STD"]

    cleaned["HAS_ANOMALY_LABEL"] = (
        anomaly_values.ne("")
    ).astype("int8")


    # --------------------------------------------------------
    # 9. Create numeric quality flags
    # --------------------------------------------------------

    cleaned["INVALID_AMOUNT_FLAG"] = (
        cleaned["USD_AMOUNT_NUM"].isna()
        &
        cleaned["USD_AMOUNT"].str.strip().ne("")
    ).astype("int8")

    cleaned["NEGATIVE_AMOUNT_FLAG"] = (
        cleaned["USD_AMOUNT_NUM"] < 0
    ).astype("int8")

    cleaned["INVALID_DURATION_FLAG"] = (
        cleaned["TOTAL_DURATION_NUM"].isna()
        &
        cleaned["TOTAL_DURATION"].str.strip().ne("")
    ).astype("int8")

    cleaned["NEGATIVE_DURATION_FLAG"] = (
        cleaned["TOTAL_DURATION_NUM"] < 0
    ).astype("int8")


    # --------------------------------------------------------
    # 10. Identify undocumented anomaly categories
    # --------------------------------------------------------

    anomaly_known = anomaly_values.isin(
        EXPECTED_CATEGORIES["ANOMALY_TYPE"]
    )

    cleaned["UNDOCUMENTED_ANOMALY_FLAG"] = (
        anomaly_values.ne("")
        &
        ~anomaly_known
    ).astype("int8")


    # --------------------------------------------------------
    # 11. Preserve the original column order
    # --------------------------------------------------------

    derived_columns = [
        col
        for col in cleaned.columns
        if col not in ORIGINAL_COLUMNS
    ]

    cleaned = cleaned[
        ORIGINAL_COLUMNS + derived_columns
    ]


    return cleaned


print("Safe cleaning function defined successfully.")

4 — Test Cleaning Function on a Small Sample

In [ ]:
# ============================================================
# CELL 4 - TEST CLEANING FUNCTION
# Purpose:
# Test derived features on a small sample before processing
# the complete dataset.
# ============================================================

# Read the first 100 records as strings
sample_raw = pd.read_csv(
    RAW_CSV_PATH,
    encoding=CSV_ENCODING,
    dtype=str,
    keep_default_na=False,
    na_filter=False,
    nrows=100
)


# ------------------------------------------------------------
# 1. Run the cleaning function
# ------------------------------------------------------------

sample_cleaned = clean_payment_chunk(
    sample_raw,
    starting_row=2
)


# ------------------------------------------------------------
# 2. Verify original data preservation
# ------------------------------------------------------------

original_preserved = (
    sample_raw[ORIGINAL_COLUMNS]
    .equals(sample_cleaned[ORIGINAL_COLUMNS])
)

assert original_preserved, (
    "Original columns were modified during cleaning."
)

assert len(sample_raw) == len(sample_cleaned), (
    "Sample row count changed."
)


# ------------------------------------------------------------
# 3. Display sample statistics
# ------------------------------------------------------------

print("=" * 65)
print("CLEANING FUNCTION - SAMPLE VALIDATION")
print("=" * 65)

print("Original records:", len(sample_raw))
print("Cleaned records:", len(sample_cleaned))

print("Original columns:", len(sample_raw.columns))
print("Cleaned columns:", len(sample_cleaned.columns))

print("Original values preserved:", original_preserved)


# ------------------------------------------------------------
# 4. Display important derived features
# ------------------------------------------------------------

columns_to_display = [
    "SOURCE_ROW_NUMBER",
    "CURRENCY_CODE",
    "CURRENCY_CODE_STD",
    "TRANSACTION_TYPE_CODE",
    "TRANSACTION_TYPE_CODE_STD",
    "PARTY_PAIR",
    "PARTICIPANT_COUNT_NUM",
    "FX_COUNT_NUM",
    "TOTAL_DURATION_NUM",
    "HAS_PATH_GAP",
    "OBSERVED_BIC_COUNT"
]

display(
    sample_cleaned[columns_to_display].head(10)
)


# ------------------------------------------------------------
# 5. Final sample validation
# ------------------------------------------------------------

print("\nSample cleaning validation passed.")

5 — Process the Complete Dataset in Chunks

In [ ]:
# ============================================================
# CELL 5 - PROCESS THE COMPLETE DATASET
# Purpose:
# Process the complete dataset in batches without deleting
# rows or modifying any original business fields.
# ============================================================

# ------------------------------------------------------------
# 1. Initialize counters
# ------------------------------------------------------------

total_input_rows = 0
total_output_rows = 0

chunk_count = 0

cleaning_stats = Counter()

standardization_changes = Counter()

first_chunk = True


# ------------------------------------------------------------
# 2. Remove an incomplete temporary output if it exists
# ------------------------------------------------------------

if TEMP_CSV_PATH.exists():
    TEMP_CSV_PATH.unlink()


# ------------------------------------------------------------
# 3. Read the original CSV in chunks
# ------------------------------------------------------------

reader = pd.read_csv(
    RAW_CSV_PATH,
    encoding=CSV_ENCODING,
    dtype=str,
    keep_default_na=False,
    na_filter=False,
    chunksize=CHUNK_SIZE
)


# ------------------------------------------------------------
# 4. Process each chunk
# ------------------------------------------------------------

for chunk_number, raw_chunk in enumerate(reader, start=1):

    chunk_count += 1

    original_values = raw_chunk[
        ORIGINAL_COLUMNS
    ].copy()

    current_chunk_size = len(raw_chunk)


    # --------------------------------------------------------
    # Apply safe cleaning
    # --------------------------------------------------------

    cleaned_chunk = clean_payment_chunk(
        raw_chunk,
        starting_row=total_input_rows + 2
    )


    # --------------------------------------------------------
    # Verify original data preservation
    # --------------------------------------------------------

    assert cleaned_chunk[
        ORIGINAL_COLUMNS
    ].equals(original_values), (
        f"Original values modified in chunk {chunk_number}."
    )

    assert len(cleaned_chunk) == current_chunk_size, (
        f"Row count changed in chunk {chunk_number}."
    )


    # --------------------------------------------------------
    # Count actual standardization differences
    # --------------------------------------------------------

    for col in STANDARDIZE_COLUMNS:

        differences = (
            cleaned_chunk[col]
            != cleaned_chunk[col + "_STD"]
        )

        standardization_changes[col] += int(
            differences.sum()
        )


    # --------------------------------------------------------
    # Collect cleaning statistics
    # --------------------------------------------------------

    cleaning_stats["PATH_GAPS"] += int(
        cleaned_chunk["HAS_PATH_GAP"].eq("Y").sum()
    )

    cleaning_stats["UNLABELED_ANOMALIES"] += int(
        cleaned_chunk["HAS_ANOMALY_LABEL"].eq(0).sum()
    )

    cleaning_stats["LABELED_ANOMALIES"] += int(
        cleaned_chunk["HAS_ANOMALY_LABEL"].eq(1).sum()
    )

    cleaning_stats["UNDOCUMENTED_ANOMALIES"] += int(
        cleaned_chunk["UNDOCUMENTED_ANOMALY_FLAG"].sum()
    )

    cleaning_stats["INVALID_AMOUNTS"] += int(
        cleaned_chunk["INVALID_AMOUNT_FLAG"].sum()
    )

    cleaning_stats["NEGATIVE_AMOUNTS"] += int(
        cleaned_chunk["NEGATIVE_AMOUNT_FLAG"].sum()
    )

    cleaning_stats["INVALID_DURATIONS"] += int(
        cleaned_chunk["INVALID_DURATION_FLAG"].sum()
    )

    cleaning_stats["NEGATIVE_DURATIONS"] += int(
        cleaned_chunk["NEGATIVE_DURATION_FLAG"].sum()
    )


    # --------------------------------------------------------
    # Write the processed chunk to a temporary CSV
    # --------------------------------------------------------

    cleaned_chunk.to_csv(
        TEMP_CSV_PATH,
        mode="w" if first_chunk else "a",
        index=False,
        header=first_chunk,
        encoding=(
            "utf-8-sig" if first_chunk else "utf-8"
        ),
        quoting=csv.QUOTE_MINIMAL,
        lineterminator="\n"
    )

    first_chunk = False


    # --------------------------------------------------------
    # Update processing counters
    # --------------------------------------------------------

    total_input_rows += current_chunk_size
    total_output_rows += len(cleaned_chunk)


    # --------------------------------------------------------
    # Display progress every 10 chunks
    # --------------------------------------------------------

    if chunk_number % 10 == 0:
        print(
            f"Chunk {chunk_number}: "
            f"{total_output_rows:,} records processed"
        )


# ------------------------------------------------------------
# 5. Verify row counts
# ------------------------------------------------------------

assert total_input_rows == total_output_rows, (
    "Input and output record counts do not match."
)


# ------------------------------------------------------------
# 6. Display processing summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("FULL DATASET PROCESSING SUMMARY")
print("=" * 65)

print("Chunks processed:", chunk_count)
print("Input records:", f"{total_input_rows:,}")
print("Output records:", f"{total_output_rows:,}")

print("Path gaps:", f"{cleaning_stats['PATH_GAPS']:,}")

print(
    "Undocumented anomaly records:",
    f"{cleaning_stats['UNDOCUMENTED_ANOMALIES']:,}"
)

print("\nOriginal values preserved during processing.")
print("Temporary cleaned file:", TEMP_CSV_PATH)

print("\nProcessing completed. Final validation is required.")

6 — Verify Exported Data Integrity

In [ ]:
# ============================================================
# CELL 6 - VERIFY CLEANED DATA INTEGRITY
# Purpose:
# Compare every original field between the raw CSV and the
# exported cleaned CSV, then publish the verified output.
# ============================================================

# ------------------------------------------------------------
# 1. Initialize validation counters
# ------------------------------------------------------------

verified_rows = 0
verified_chunks = 0
mismatched_cells = 0

validation_passed = True


# ------------------------------------------------------------
# 2. Create readers for both files
# ------------------------------------------------------------

raw_reader = pd.read_csv(
    RAW_CSV_PATH,
    encoding=CSV_ENCODING,
    dtype=str,
    keep_default_na=False,
    na_filter=False,
    chunksize=CHUNK_SIZE
)

clean_reader = pd.read_csv(
    TEMP_CSV_PATH,
    encoding=CSV_ENCODING,
    dtype=str,
    keep_default_na=False,
    na_filter=False,
    chunksize=CHUNK_SIZE
)


# ------------------------------------------------------------
# 3. Compare all original columns
# ------------------------------------------------------------

for raw_chunk, clean_chunk in zip_longest(
    raw_reader,
    clean_reader
):

    verified_chunks += 1

    if raw_chunk is None or clean_chunk is None:
        validation_passed = False
        print("Validation failed: Chunk count mismatch.")
        break

    if len(raw_chunk) != len(clean_chunk):
        validation_passed = False
        print("Validation failed: Row count mismatch.")
        break

    original_data = raw_chunk[ORIGINAL_COLUMNS]

    cleaned_original_data = clean_chunk[
        ORIGINAL_COLUMNS
    ]


    # --------------------------------------------------------
    # Compare every original cell value
    # --------------------------------------------------------

    if not original_data.equals(cleaned_original_data):

        validation_passed = False

        differences = (
            original_data.to_numpy()
            != cleaned_original_data.to_numpy()
        )

        mismatched_cells += int(differences.sum())

        print(
            f"Validation failed in chunk {verified_chunks}."
        )

        break

    verified_rows += len(raw_chunk)


# ------------------------------------------------------------
# 4. Check final validation conditions
# ------------------------------------------------------------

assert validation_passed, (
    "Cleaned dataset failed the integrity validation."
)

assert verified_rows == total_input_rows, (
    "Verified row count does not match the source."
)

assert mismatched_cells == 0, (
    "Original data mismatch detected."
)


# ------------------------------------------------------------
# 5. Publish the validated dataset
# ------------------------------------------------------------

os.replace(
    TEMP_CSV_PATH,
    CLEAN_CSV_PATH
)


# ------------------------------------------------------------
# 6. Display validation results
# ------------------------------------------------------------

print("=" * 65)
print("CLEANED DATA INTEGRITY VALIDATION")
print("=" * 65)

print("Records verified:", f"{verified_rows:,}")
print("Chunks verified:", verified_chunks)
print("Mismatched original cells:", mismatched_cells)

print("Validation passed:", validation_passed)

print("\nOriginal columns preserved: True")
print("Record order preserved: True")

print("\nExport integrity validation passed.")
print("Verified cleaned dataset:", CLEAN_CSV_PATH)

7 — Generate Cleaning Log

In [ ]:
# ============================================================
# CELL 7 - GENERATE CLEANING LOG
# Purpose:
# Document transformations, preservation decisions, and
# actual standardization counts.
# ============================================================

cleaning_actions = []


# ------------------------------------------------------------
# 1. Record original data preservation
# ------------------------------------------------------------

cleaning_actions.append({
    "FIELD": "ALL_ORIGINAL_COLUMNS",
    "ACTION": "Preserved original values",
    "AFFECTED_RECORDS": total_input_rows,
    "REASON": "Maintain original business information",
    "ORIGINAL_DATA_MODIFIED": "NO"
})


# ------------------------------------------------------------
# 2. Record categorical standardization
# ------------------------------------------------------------

for col in STANDARDIZE_COLUMNS:

    cleaning_actions.append({
        "FIELD": col,
        "ACTION": "Created standardized analysis column",
        "AFFECTED_RECORDS": standardization_changes[col],
        "REASON": (
            "Normalize surrounding whitespace and casing "
            "in derived categorical fields only"
        ),
        "ORIGINAL_DATA_MODIFIED": "NO"
    })


# ------------------------------------------------------------
# 3. Record numeric conversions
# ------------------------------------------------------------

for original_col, derived_col in NUMERIC_MAPPING.items():

    cleaning_actions.append({
        "FIELD": original_col,
        "ACTION": f"Created numeric analysis field {derived_col}",
        "AFFECTED_RECORDS": total_input_rows,
        "REASON": "Enable statistical calculations",
        "ORIGINAL_DATA_MODIFIED": "NO"
    })


# ------------------------------------------------------------
# 4. Record derived features
# ------------------------------------------------------------

derived_feature_actions = [
    (
        "EPOCH_SECONDS",
        "Derived UTC date and time fields",
        "Support temporal analysis"
    ),
    (
        "ORDERING_TYPE / BENEFICIARY_TYPE",
        "Derived PARTY_PAIR",
        "Support business relationship analysis"
    ),
    (
        "BIC8_PATH",
        "Derived HAS_PATH_GAP and OBSERVED_BIC_COUNT",
        "Support routing analysis"
    ),
    (
        "ANOMALY_TYPE",
        "Derived anomaly label indicators",
        "Support anomaly analysis"
    )
]

for field, action, reason in derived_feature_actions:

    cleaning_actions.append({
        "FIELD": field,
        "ACTION": action,
        "AFFECTED_RECORDS": total_input_rows,
        "REASON": reason,
        "ORIGINAL_DATA_MODIFIED": "NO"
    })


# ------------------------------------------------------------
# 5. Record important preservation decisions
# ------------------------------------------------------------

preservation_actions = [
    (
        "BIC8_PATH",
        "Preserved routing gaps",
        cleaning_stats["PATH_GAPS"],
        "Routing gaps may be generated business behaviour"
    ),
    (
        "ANOMALY_TYPE",
        "Preserved undocumented anomaly categories",
        cleaning_stats["UNDOCUMENTED_ANOMALIES"],
        "Undocumented categories require further investigation"
    ),
    (
        "ANOMALY_TYPE",
        "Preserved blank anomaly labels",
        cleaning_stats["UNLABELED_ANOMALIES"],
        "Blank values must not be assumed to mean normal"
    ),
    (
        "USD_AMOUNT",
        "Preserved extreme amounts",
        total_input_rows,
        "Potential anomaly-related amount patterns"
    )
]

for field, action, count, reason in preservation_actions:

    cleaning_actions.append({
        "FIELD": field,
        "ACTION": action,
        "AFFECTED_RECORDS": count,
        "REASON": reason,
        "ORIGINAL_DATA_MODIFIED": "NO"
    })


# ------------------------------------------------------------
# 6. Create and export the log
# ------------------------------------------------------------

cleaning_log = pd.DataFrame(cleaning_actions)

cleaning_log.to_csv(
    CLEANING_LOG_PATH,
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# 7. Display the results
# ------------------------------------------------------------

print("=" * 65)
print("DATA CLEANING ACTIONS LOG")
print("=" * 65)

print("Total logged actions:", len(cleaning_log))
print("Log location:", CLEANING_LOG_PATH)

display(cleaning_log)

print("\nCleaning log generated successfully.")

8 — Compare Results Against the Data Audit

In [ ]:
# ============================================================
# CELL 8 - CROSS-CHECK CLEANING AGAINST DATA AUDIT
# Purpose:
# Ensure the processed dataset preserves key findings from
# the original data quality audit.
# ============================================================

AUDIT_DIR = PROJECT_ROOT / "outputs" / "tables"

AUDIT_SUMMARY_PATH = (
    AUDIT_DIR / "audit_summary.csv"
)

AUDIT_COLUMN_PATH = (
    AUDIT_DIR / "audit_column_profile.csv"
)

AUDIT_CATEGORY_PATH = (
    AUDIT_DIR / "audit_category_profile.csv"
)

AUDIT_UNEXPECTED_PATH = (
    AUDIT_DIR / "audit_unexpected_categories.csv"
)


# ------------------------------------------------------------
# 1. Load previous audit results
# ------------------------------------------------------------

audit_summary_df = pd.read_csv(
    AUDIT_SUMMARY_PATH
)

audit_column_df = pd.read_csv(
    AUDIT_COLUMN_PATH
)

audit_category_df = pd.read_csv(
    AUDIT_CATEGORY_PATH,
    keep_default_na=False
)

audit_unexpected_df = pd.read_csv(
    AUDIT_UNEXPECTED_PATH,
    keep_default_na=False
)


# ------------------------------------------------------------
# 2. Extract audit values
# ------------------------------------------------------------

def get_audit_metric(metric_name):

    result = audit_summary_df.loc[
        audit_summary_df["METRIC"] == metric_name,
        "VALUE"
    ]

    assert len(result) == 1, (
        f"Audit metric not found: {metric_name}"
    )

    return int(result.iloc[0])


expected_rows = get_audit_metric(
    "Total records"
)

expected_path_gaps = get_audit_metric(
    "Routing paths with gaps"
)

expected_blank_anomalies = int(
    audit_column_df.loc[
        audit_column_df["COLUMN"] == "ANOMALY_TYPE",
        "BLANK_COUNT"
    ].iloc[0]
)

expected_undocumented = int(
    audit_unexpected_df["COUNT"].sum()
)


# ------------------------------------------------------------
# 3. Compare audit and cleaning values
# ------------------------------------------------------------

audit_comparison = pd.DataFrame([
    {
        "METRIC": "Total records",
        "AUDIT_VALUE": expected_rows,
        "CLEANING_VALUE": total_output_rows
    },
    {
        "METRIC": "Routing path gaps",
        "AUDIT_VALUE": expected_path_gaps,
        "CLEANING_VALUE": cleaning_stats["PATH_GAPS"]
    },
    {
        "METRIC": "Blank anomaly labels",
        "AUDIT_VALUE": expected_blank_anomalies,
        "CLEANING_VALUE": cleaning_stats["UNLABELED_ANOMALIES"]
    },
    {
        "METRIC": "Undocumented anomaly categories",
        "AUDIT_VALUE": expected_undocumented,
        "CLEANING_VALUE": cleaning_stats["UNDOCUMENTED_ANOMALIES"]
    }
])


# ------------------------------------------------------------
# 4. Validate comparisons
# ------------------------------------------------------------

audit_comparison["MATCH"] = (
    audit_comparison["AUDIT_VALUE"]
    ==
    audit_comparison["CLEANING_VALUE"]
)

display(audit_comparison)

assert audit_comparison["MATCH"].all(), (
    "Cleaning results do not match the original audit."
)


# ------------------------------------------------------------
# 5. Export comparison results
# ------------------------------------------------------------

audit_comparison.to_csv(
    REPORT_DIR / "audit_cleaning_comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Audit-to-cleaning comparison passed.")

9 — Generate Final Validation Summary

In [ ]:
# ============================================================
# CELL 9 - GENERATE FINAL VALIDATION SUMMARY
# Purpose:
# Document the final cleaning status and integrity checks
# for reproducibility and competition reporting.
# ============================================================

assert CLEAN_CSV_PATH.is_file(), (
    "Validated cleaned dataset was not found."
)

assert validation_passed
assert audit_comparison["MATCH"].all()


# ------------------------------------------------------------
# 1. Generate validation report
# ------------------------------------------------------------

validation_results = [
    {
        "CHECK": "Original data columns preserved",
        "RESULT": "PASS",
        "DETAILS": "All 21 original columns verified"
    },
    {
        "CHECK": "Total records preserved",
        "RESULT": "PASS",
        "DETAILS": str(total_output_rows)
    },
    {
        "CHECK": "Original cell values preserved",
        "RESULT": "PASS",
        "DETAILS": "No mismatches in original columns"
    },
    {
        "CHECK": "Record order preserved",
        "RESULT": "PASS",
        "DETAILS": "Sequential chunk comparison"
    },
    {
        "CHECK": "Routing gaps preserved",
        "RESULT": "PASS",
        "DETAILS": str(cleaning_stats["PATH_GAPS"])
    },
    {
        "CHECK": "Undocumented anomaly categories preserved",
        "RESULT": "PASS",
        "DETAILS": str(
            cleaning_stats["UNDOCUMENTED_ANOMALIES"]
        )
    },
    {
        "CHECK": "Audit consistency",
        "RESULT": "PASS",
        "DETAILS": "All selected audit metrics matched"
    },
    {
        "CHECK": "Original Excel to CSV verification",
        "RESULT": "PARTIAL",
        "DETAILS": (
            "Excel row counts matched and the first 100 "
            "records were sample-checked. Full Excel-to-CSV "
            "cell-level reconciliation was not performed."
        )
    }
]

validation_df = pd.DataFrame(validation_results)


# ------------------------------------------------------------
# 2. Export validation report
# ------------------------------------------------------------

validation_df.to_csv(
    VALIDATION_REPORT_PATH,
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# 3. Display final report
# ------------------------------------------------------------

print("=" * 65)
print("SWIFT DATATHON - FINAL DATA CLEANING REPORT")
print("=" * 65)

display(validation_df)

print("\nOriginal transaction records:", f"{total_input_rows:,}")
print("Cleaned transaction records:", f"{total_output_rows:,}")

print("\nOriginal columns:", len(ORIGINAL_COLUMNS))

print(
    "Derived columns:",
    len(
        pd.read_csv(
            CLEAN_CSV_PATH,
            nrows=0
        ).columns
    ) - len(ORIGINAL_COLUMNS)
)

print("\nCleaned Dataset:")
print(CLEAN_CSV_PATH)

print("\nCleaning Log:")
print(CLEANING_LOG_PATH)

print("\nValidation Report:")
print(VALIDATION_REPORT_PATH)

print("\nDATA CLEANING COMPLETED SUCCESSFULLY.")
print("The original business fields remain unchanged.")